# Creative Advertising Multi-Agent System

**Assignment:** Week 2 — Part B  
**Framework:** OpenAI Agents SDK  
**Pipeline:** Creative Director → Strategist → Copywriter

---

In [ ]:
# Section 1a — Install dependencies (run once)
!pip install openai-agents python-dotenv nest_asyncio --quiet

In [ ]:
# Section 1b — Imports
import asyncio
import os
import nest_asyncio
from dotenv import load_dotenv
from agents import Agent, Runner

# Patch Jupyter's event loop so asyncio.run() works inside cells
nest_asyncio.apply()

print("Imports OK")

In [ ]:
# Section 1c — Load API key from .env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Copy .env.example to .env and add your key.")
print("API key loaded.")

In [ ]:
# Section 2 — Define the three agents

creative_director = Agent(
    name="Creative Director",
    model="gpt-4o-mini",
    instructions=(
        "You are a Creative Director at a top advertising agency. "
        "Given a product launch brief, generate 3 to 5 distinct campaign ideas. "
        "Format each idea with: **Name** (bold), *Tagline* (italic), "
        "and a 2-sentence description of the concept and target audience."
    ),
)

strategist = Agent(
    name="Strategist",
    model="gpt-4o-mini",
    instructions=(
        "You are a Marketing Strategist. You will receive a list of campaign ideas. "
        "Review them and select the top 2 ideas based on market potential, "
        "cultural fit, and originality. "
        "For each selected idea, explain in 2-3 sentences why it was chosen "
        "and what makes it commercially strong."
    ),
)

copywriter = Agent(
    name="Copywriter",
    model="gpt-4o-mini",
    instructions=(
        "You are a social media Copywriter. You will receive two selected campaign concepts. "
        "For each campaign, write exactly 3 tweets. "
        "Each tweet must: be under 280 characters, include 2-3 relevant hashtags, "
        "have an inspiring and eco-conscious tone, "
        "and feel native to the target location and audience."
    ),
)

print("Agents defined: Creative Director, Strategist, Copywriter")

In [ ]:
# Section 2 (cont.) — Pipeline function

async def run_pipeline(prompt: str):
    """Run the 3-agent campaign pipeline. Returns (ideas, strategy, tweets)."""
    cd_result = await Runner.run(creative_director, prompt)
    ideas = cd_result.final_output

    st_result = await Runner.run(strategist, ideas)
    strategy = st_result.final_output

    cw_result = await Runner.run(copywriter, strategy)
    tweets = cw_result.final_output

    return ideas, strategy, tweets

print("Pipeline function defined.")

In [ ]:
# Section 3a — Step 1: Run Creative Director
TEST_PROMPT = "Launch a campaign for a new eco-friendly water bottle in Bali."

print("Running Creative Director...\n")
cd_result = asyncio.run(Runner.run(creative_director, TEST_PROMPT))
ideas = cd_result.final_output

print("=" * 60)
print("STEP 1: CREATIVE DIRECTOR — Campaign Ideas")
print("=" * 60)
print(ideas)

In [ ]:
# Section 3b — Step 2: Run Strategist
print("Running Strategist...\n")
st_result = asyncio.run(Runner.run(strategist, ideas))
strategy = st_result.final_output

print("=" * 60)
print("STEP 2: STRATEGIST — Top 2 Selections & Reasoning")
print("=" * 60)
print(strategy)

In [ ]:
# Section 3c — Step 3: Run Copywriter
print("Running Copywriter...\n")
cw_result = asyncio.run(Runner.run(copywriter, strategy))
tweets = cw_result.final_output

print("=" * 60)
print("STEP 3: COPYWRITER — Tweets")
print("=" * 60)
print(tweets)

## Section 4 — Sample Output

> Run all cells above (Kernel → Restart & Run All) to generate live output.
> After running, the cell outputs above serve as the sample output for submission.

*(The frozen cell outputs above are produced by the test prompt: "Launch a campaign for a new eco-friendly water bottle in Bali.")*

## Section 5 — System Explanation

### Role of Each Agent

| Agent | Role | Responsibility |
|---|---|---|
| **Creative Director** | Divergent thinker | Generates 3–5 distinct campaign ideas, each with a name, tagline, and 2-sentence description |
| **Strategist** | Convergent thinker | Reviews all ideas and selects the top 2 based on market potential, cultural fit, and originality |
| **Copywriter** | Executor | Translates each selected campaign into 3 ready-to-post tweets with hashtags |

### Tools and Functions Used

- **`Agent(name, model, instructions)`** — Defines each agent's persona, the LLM model it runs on, and its task instructions. No tools or function calls are used — these are pure text-in/text-out agents.
- **`Runner.run(agent, input)`** — Executes an agent asynchronously given an input string. Returns a `RunResult` object containing the agent's response.
- **`result.final_output`** — A string attribute on `RunResult` that holds the agent's final text response. This string is passed directly as the input to the next agent in the pipeline.
- **`nest_asyncio.apply()`** — Patches Python's running event loop so that `asyncio.run()` can be called from inside Jupyter notebook cells (which already have a running event loop).
- **`python-dotenv` / `load_dotenv()`** — Loads environment variables from a `.env` file into `os.environ`, allowing the `OPENAI_API_KEY` to be set without hardcoding secrets in the notebook.

### Why a Multi-Agent Approach Improves the Workflow

A single large prompt asking one LLM to generate ideas, evaluate them, and write tweets in one pass produces mediocre results — the model cannot switch creative modes mid-response without quality loss.

The multi-agent pipeline improves on this in four concrete ways:

1. **Focused roles** — Each agent receives instructions optimized for exactly one task. The Creative Director is not constrained by commercial concerns; the Strategist is not biased toward ideas it generated itself.
2. **Real-team mirroring** — The Creative Director → Strategist → Copywriter sequence maps to how human advertising teams work. This makes the system's reasoning transparent and easy to audit.
3. **Inspectable intermediate outputs** — Each agent's output is a visible string before it enters the next stage. A human can read, edit, or redirect it between steps without restarting the pipeline.
4. **Modularity** — Any agent can be updated independently. Changing the Copywriter's tone (e.g., to LinkedIn format) requires editing only that agent's instructions, with no effect on the others.
